# Data Visualization: Time and Stepwise Change
## Class 1 — Time Series Line Plot + Candlestick Chart

Today we explore two techniques for visualizing data that changes over time:
- **Time Series Line Plots** — trends, rolling averages, seasonality
- **Candlestick Charts** — OHLC financial data, volatility interpretation

## Learning Objectives
- Parse and index date columns correctly in pandas
- Plot annotated time series with trend lines and rolling averages
- Identify and highlight seasonality in time data
- Construct a candlestick chart using `mplfinance`
- Read OHLC patterns and interpret basic volatility signals


In [1]:
# ── Environment Setup ──────────────────────────────────────────────────────
# pandas     → tabular data + DatetimeIndex handling
# numpy      → numerical operations
# matplotlib → base plotting engine; mdates handles date axis formatting
# seaborn    → statistical plots and color palettes

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates      # date tick locators and formatters
import matplotlib.ticker as mticker    # value axis formatters ($ sign, commas)
import seaborn as sns
from datetime import datetime

# mplfinance: professional financial chart library (candlesticks)
try:   #used for error handling.
    import mplfinance as mpf
    MPLFINANCE = True    # A variable is created to indicate that the library exists.
    print("mplfinance available — candlestick charts will use mpf.plot()")
except ImportError:      # Runs this block if mplfinance cannot be imported.
    MPLFINANCE = False   # Sets a flag showing mplfinance is not installed.
    print("mplfinance not found. Run:  pip install mplfinance")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (13, 6)  # controls global matplotlib settings, sets default plot size.
%matplotlib inline                        # Without it, plots may open in a separate window.
print("\nEnvironment ready!")


mplfinance not found. Run:  pip install mplfinance


UsageError: unrecognized arguments: # Without it, plots may open in a separate window.


---
# Part 1: Time Series Line Plots

A **time series** is a sequence of observations indexed in chronological order.
Line plots connect successive time steps, making trends and cycles immediately visible.

## Why Date Handling Matters
Raw data often stores dates as strings (`"2024-01-15"`).
Before plotting, we must:
1. **Parse** strings → `datetime` objects: `pd.to_datetime(col)`
2. **Set index** → `DatetimeIndex`: `df.set_index("date_col")`
3. **Sort** → chronological order: `df.sort_index()`

## Key Concepts We Will Cover
1. Basic time series plot with proper date axis formatting
2. Rolling averages — smoothing noise to reveal underlying trend
3. Seasonality markers — shading and annotating recurring patterns
4. Common mistakes and corrections


In [ ]:
# ── Dataset: Monthly E-Commerce Revenue (3 Years) ─────────────────────────
# We simulate 36 months of revenue with:
#   - A rising linear trend ($80k → $160k)  Revenue will gradually increase from $80,000 to $160,000.
#   - Holiday season spikes in November and December Black Friday, Christmas, New Year sales
#   - Random month-to-month noise ±8%. Random variation of about ±8% will be added to simulate realistic data.
#    Real data is not perfectly smooth. There are random fluctuations.

np.random.seed(42)    # With seed → same dataset every run

# pd.date_range with freq="MS" = Month Start — first day of each month
# Generates a sequence of monthly dates from Jan-2022 to Dec-2024.
# start = 2022-01-01 , end   = 2024-12-01 , freq  = MS (month start)
dates = pd.date_range(start="2022-01-01", end="2024-12-01", freq="MS")        # generates a sequence of dates.

# Stores the total number of months in the dataset.
n = len(dates)   # Count Number of Months, 3 years × 12 months = 36 months

# Base trend: create a linear revenue growth trend
base_trend = np.linspace(80_000, 160_000, n)      # Creates a steadily increasing revenue trend from $80k to $160k, n= generates evenly spaced numbers.

# Seasonal multipliers: Nov/Dec get +35%, Jun/Jul get +10%
seasonal_boost = np.array([               # This array will store seasonal multipliers for each month.
    1.35 if m in (11, 12) else (1.10 if m in (6, 7) else 1.0)    #if month is Nov or Dec → 1.35 else if month is Jun or Jul → 1.10 else → 1.0
    for m in dates.month
])
# Assigns a seasonal multiplier depending on the month.
# Revenue × 1.35 → holiday boost , Revenue × 1.10 → summer boost , Revenue × 1.00 → normal

# Gaussian noise: realistic random variation
noise = np.random.normal(0, base_trend * 0.08)

# Final revenue signal
revenue = (base_trend * seasonal_boost) + noise

# Create DataFrame — DatetimeIndex is set automatically by pd.date_range
df_rev = pd.DataFrame({"Revenue": revenue.round(0)}, index=dates)
df_rev.index.name = "Date"

print("Dataset preview (first 12 months):")
print(df_rev.head(12).to_string())
print(f"\nTotal months : {len(df_rev)}")
print(f"Date range   : {df_rev.index[0].date()} → {df_rev.index[-1].date()}")
print(f"Min revenue  : ${df_rev['Revenue'].min():>10,.0f}")
print(f"Max revenue  : ${df_rev['Revenue'].max():>10,.0f}")
print(f"Mean revenue : ${df_rev['Revenue'].mean():>10,.0f}")


In [ ]:
#We already created a time-series dataset of monthly revenue.
#how revenue changes over time , if revenue is increasing , if there are seasonal spikes
#Convert raw data → a readable time-series chart.

# ── Basic Time Series Line Plot ────────────────────────────────────────────
# Step-by-step: from raw data to a properly formatted chart.

fig, ax = plt.subplots(figsize=(13, 5))

# ax.plot() with a DatetimeIndex auto-formats the X axis as dates, ax.plot() connects points using a line.
# marker="o" adds circles at each monthly observation
# the actual time-series line plot.
ax.plot(df_rev.index, df_rev["Revenue"],    # the date colum(Jan 2022, Feb 2022), the revenue value(80k, 82k)
        color="steelblue", linewidth=1.8,
        marker="o", markersize=4, label="Monthly Revenue")   # Date → X-axis, Revenue → Y-axis

# ── Date Axis Formatting ────────────────────────────────────────────────────
# mdates.MonthLocator(interval=3) → place a tick every 3 months
# mdates.DateFormatter("%b %Y")   → label as "Jan 2022", "Apr 2022", etc.
# Without formatting, the axis might show too many dates.
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3)) # show one label every 3 months. Places a tick mark on the x-axis every 3 months.

# Formats date labels as "Month Year".
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y")) # %b = short month name, this controls how the date text appears(Jan 2022 Apr 2022)

# Rotates date labels slightly so they do not overlap.
plt.xticks(rotation=30, ha="right")  # If labels are horizontal, they may overlap. Tilts the labels 30 degrees.

# ── Y-axis: show dollar signs with comma separator ────────────────────────── Without formatting: 80000, After formatting: $80,000
# mticker.FuncFormatter takes a function f(value, position) → string
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"${x:,.0f}") #lambda x means value on the axis, .0f → no decimals
)

ax.set_title("Monthly E-Commerce Revenue (Jan 2022 – Dec 2024)", fontsize=15, pad=14)
ax.set_xlabel("Month", fontsize=12)
ax.set_ylabel("Revenue ($)", fontsize=12)
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Observations from the raw line:")
print("  • Clear upward trend across the 3 years")
print("  • Visible spikes in late-year months — holiday season effect")
print("  • Month-to-month noise makes quantifying the trend difficult")
print("  → Next: rolling averages will smooth the noise")


In [ ]:
# In the previous graph we saw raw monthly revenue.
#But real data always contains: trend (long-term growth), seasonality (holiday spikes), noise (random fluctuations)

# problem: Because of noise, it is hard to clearly see the real trend.
# Use: A rolling average smooths the data. Instead of using one month, we average multiple months together.
# Instead of looking at one month, we look at multiple months together and take their average.

# ── Rolling Averages — Trend Extraction ────────────────────────────────────
# rolling(window=N).mean() computes the mean over a sliding N-period window. The window slides forward each month.

# This removes small fluctuations and shows the underlying trend.

# Short window → more responsive to recent changes (still noisy)
# Long window  → smoother trend line (lags behind sharp changes)

# Rolling average means: Take a few nearby values, calculate their average, then move forward and repeat. It is used to smooth noisy data and reveal the trend.

# Two Types of Rolling Averages Used
# 3-Month Rolling Average. Short window. Shows short-term trend. Still reacts quickly to changes.
# 12-Month Rolling Average. Large window. Shows long-term yearly trend. Very smooth but slower to react.

# 3-month rolling average: short-term smoothing. The window slides forward one step each time. Create a moving window containing N values.
# rolling(window=N).mean() computes the mean over a sliding N-period window

df_rev["MA_3"]  = df_rev["Revenue"].rolling(window=3).mean()  #Rolling (Window 1= Jan Feb Mar)

# 12-month rolling average: full-year trend (the 'true' annual trajectory). Each value becomes the average of the last 12 months.
df_rev["MA_12"] = df_rev["Revenue"].rolling(window=12).mean()         # This shows the true yearly trend.

# NOTE: The first (window-1) values will be NaN because the window is not yet full.
# MA_3  → first 2 rows are NaN
# MA_12 → first 11 rows are NaN  (this is expected and correct) Rolling average needs full window data. If only 6 or 9 months exist, the average cannot be calculated.
print("First 14 rows showing NaN fill-in period:")

# Displays first 14 rows to show how rolling averages start with NaN values.

print(df_rev[["Revenue","MA_3","MA_12"]].head(14).to_string()) #Selects these three columns. Selects these three columns. Prints the table to_string()

fig, ax = plt.subplots(figsize=(13, 6))

# Raw data — light color, thin line; serves as context
ax.plot(df_rev.index, df_rev["Revenue"],        # Plots original monthly revenue values.
        color="lightsteelblue", linewidth=1.2,  # Uses light blue thin line for raw data.
        marker="o", markersize=3, alpha=0.8, label="Monthly Revenue (raw)")    # Adds small circular markers and label for raw revenue.

# 3-month MA — responsive short-term smoother
ax.plot(df_rev.index, df_rev["MA_3"],         # Plot 3-Month Rolling Average. Plots the 3-month rolling average.
        color="darkorange", linewidth=2.2,
        linestyle="--", label="3-Month Rolling Avg")        # Displays 3-month average as dashed orange line.


# 12-month MA — the dominant long-term trend line
ax.plot(df_rev.index, df_rev["MA_12"],                      # Plots the 12-month rolling average.
        color="crimson", linewidth=2.8,
        linestyle="-", label="12-Month Rolling Avg (Trend)")  # Shows the yearly trend using a solid red line.

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))    # Shows tick every 3 months.
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))    # Formats labels as month and year.
plt.xticks(rotation=30, ha="right")   # Rotates labels to prevent overlap.
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))  # Formats Y-axis numbers as dollar currency.

ax.set_title("Revenue with Rolling Averages — Smoothing Noise to See Trend",
             fontsize=15, pad=14)
ax.set_xlabel("Month"); ax.set_ylabel("Revenue ($)")  # Labels both axes.
ax.legend(fontsize=11); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

print("\nRolling Average Insights:")
print("  • 3-month MA: still shows seasonal peaks but much smoother than raw")  # short-term average still shows seasonality.
print("  • 12-month MA: clean long-term trend — clearly rises over 3 years")   # yearly average reveals long-term growth.
print("  • Choose window based on noise-vs-responsiveness trade-off")         # window size depends on smoothness vs responsiveness.


In [ ]:
# Seasonality

#We already saw: Trend → long-term growth (12-month moving average).......Noise → random month-to-month fluctuations
#Now we want to see recurring patterns, i.e., seasonality.

#Seasonality means: Certain times of the year always behave differently. In our dataset: holiday season (Q4: Oct–Dec) spikes revenue.

# ── Seasonality Markers — Highlighting Recurring Patterns ────────────────
# Seasonality = repeating patterns at regular calendar intervals.
# We mark Q4 (Oct–Dec) holiday windows and annotate December peaks.

#Plot revenue again, along with the 12-month trend line. Highlight the holiday season (Q4) using a colored band. Annotate the December peaks so they are easy to see.

fig, ax = plt.subplots(figsize=(13, 6))

# We saw, Raw revenue (blue line) → monthly fluctuations. 12-month trend (red dashed line) → long-term growth
# This helps: Helps us compare actual revenue vs trend. Peaks that are above the trend indicate seasonal spikes.

ax.plot(df_rev.index, df_rev["Revenue"],
        color="steelblue", linewidth=1.8, marker="o", markersize=4,
        label="Monthly Revenue")
ax.plot(df_rev.index, df_rev["MA_12"],
        color="crimson", linewidth=2.5, linestyle="--", label="12-Month Trend")

# ── Shade Q4 windows (Oct 1 → Jan 1) in gold ───────────────────────────── # Highlight Q4, We create shaded vertical bands for Q4 (Oct → Dec)
# which months are holiday months. Shading is just a visual cue for a recurring pattern. Q4 = seasonally high revenue period.

# ax.axvspan(xmin, xmax) fills a vertical band between two x dates
for year in [2022, 2023, 2024]:
    ax.axvspan(
        datetime(year, 10, 1),       # October 1st
        datetime(year + 1, 1, 1),    # January 1st of next year
        alpha=0.12, color="gold",
        label="Holiday Season (Q4)" if year == 2022 else ""  # label once only
    )

# ── Annotate each December revenue peak ──────────────────────────────────
# specifically label each December point with its revenue: Adds the exact value of revenue for December months. Makes it easy to see the year-over-year (YoY) increase.
dec_mask  = df_rev.index.month == 12
dec_dates = df_rev.index[dec_mask]
dec_vals  = df_rev["Revenue"][dec_mask]

for date, val in zip(dec_dates, dec_vals):
    ax.annotate(
        f"Dec Peak\n${val:,.0f}",   # two-line label
        xy=(date, val),              # point being annotated
        xytext=(0, 24),              # text offset: 24 pts above the data point
        textcoords="offset points",
        ha="center", fontsize=8, color="darkred", fontweight="bold",
        arrowprops=dict(arrowstyle="->", color="darkred", lw=1.2)
    )

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=30, ha="right")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))

ax.set_title("Revenue with Holiday Seasonality Highlighted", fontsize=15, pad=14)
ax.set_xlabel("Month"); ax.set_ylabel("Revenue ($)")
ax.legend(fontsize=10, loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

print("Seasonality Analysis:")
print("  • Yellow shading marks Q4 each year — holiday shopping season")
print("  • December is consistently the highest revenue month each year")
print("  ► Revenue grows YoY — both trend line and peak values rise every Dec")

# Yellow shading marks holiday season (Q4).
# December consistently has the highest revenue.
# Year-over-year, both the trend line and peak revenue increase, showing growth over time + seasonal effect.

In [ ]:
# Before plotting a time series, convert dates → datetime, set index, sort chronologically, format labels,
# and add horizontal grids to make trends and patterns clear.

# ── Common Time Series Mistakes ────────────────────────────────────────────
# Two of the most frequent problems — shown with corrections side by side. Wrong → unsorted dates, Correct → sorted dates

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── LEFT: Unsorted dates → meaningless zig-zag line ────────────────────────

#selects only the Revenue column, selects only the Revenue column, makes the shuffle reproducible
df_shuffled = df_rev[["Revenue"]].sample(frac=1, random_state=7) # Create a version of the data where dates are out of order.

#plot on the left subplot, X-axis (dates, but shuffled), X-axis (dates, but shuffled), Y-axis (revenue), red line, small circles for each data point
axes[0].plot(df_shuffled.index, df_shuffled["Revenue"],
             color="tomato", linewidth=1.5, marker="o", markersize=3)   # Plots revenue with unsorted dates, showing a meaningless zig-zag line.
# Since the dates are shuffled, the line connects points in random order → creates a zig-zag line that makes no sense

axes[0].set_title("Mistake: Unsorted Dates\n(line zig-zags — trend invisible)",
                  fontsize=12, color="crimson", pad=10)

axes[0].set_xlabel("Date"); axes[0].set_ylabel("Revenue ($)")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))     # # Shows year only on X-axis for readability.
axes[0].tick_params(axis="x", rotation=30)    # Rotates date labels to prevent clutter.

# ── RIGHT: Sorted + formatted → clean trend visible ─────────────────────────

# select Revenue column, select Revenue column
df_correct = df_rev[["Revenue"]].sort_index()          #Correct order of dates is required to see the trend.

axes[1].plot(df_correct.index, df_correct["Revenue"], ## Plots sorted revenue data on right subplot, Color = green, line thicker than mistake plot
             color="seagreen", linewidth=2, marker="o", markersize=3)  # line flows left → right, trend is visible, peaks make sense.

axes[1].set_title("Correct: Dates Sorted Ascending\n(trend flows left-to-right naturally)",
                  fontsize=12, color="green", pad=10)

axes[1].set_xlabel("Date"); axes[1].set_ylabel("")      # # Adds X-axis label to the right subplot.
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))    # Shows date ticks every 6 months for clarity

# Formats X-axis labels as "Jan 2022" instead of raw numbers
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))  # Formats date labels to be readable month + year

axes[1].tick_params(axis="x", rotation=30)
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Time Series Common Mistakes", fontsize=15, y=1.02)
plt.tight_layout(); plt.show()

print("5 Rules for Clean Time Series:")
print("  1. Parse strings → datetime:  pd.to_datetime(df['date_col'])")
print("  2. Set DatetimeIndex:          df.set_index('date_col')")
print("  3. Sort chronologically:       df.sort_index()")
print("  4. Format tick labels:         mdates.DateFormatter('%b %Y')")
print("  5. Grid on value axis only:    ax.grid(axis='y')")


In [ ]:
# Rules for Clean Time Series

#Convert string → datetime
#Example: "2024-01-15" → datetime(2024,1,15)

#Why: allows matplotlib/pandas to understand chronological order

#Set DatetimeIndex
#Makes date the main axis of the dataframe
#Useful for rolling averages, slicing, plotting
#Sort chronologically
#Always sort from earliest → latest
#Avoids zig-zag lines
#Format tick labels nicely
#Display readable months/years instead of default numbers
#Helps audience interpret dates quickly
#Grid on value axis only
#Horizontal grid lines (Y-axis) help read values
#Vertical grid lines (X-axis) are usually unnecessary

---
# Part 2: Candlestick Charts — Reading OHLC Data

A **candlestick chart** is the standard financial market visualization.
Each candle represents one time period and encodes **4 prices**:

| Letter | Name  | Meaning |
|--------|-------|---------|
| **O**  | Open  | Price at the START of the period |
| **H**  | High  | Highest price DURING the period |
| **L**  | Low   | Lowest price DURING the period |
| **C**  | Close | Price at the END of the period |

## Anatomy of a Candle
```
    │   ← Upper wick  (High − max(O, C))
  ┌─┴─┐
  │   │ ← Body (between Open and Close)
  └─┬─┘
    │   ← Lower wick  (min(O, C) − Low)
```
- **Green candle**: Close > Open → price went **UP** (bullish session)
- **Red candle**: Close < Open → price went **DOWN** (bearish session)
- **Long wicks**: high volatility — price moved far but reversed
- **Short body + long wicks**: indecision (Doji pattern)

## Key Concepts We Will Cover
1. OHLC data structure and validation
2. Candlestick chart with `mplfinance` (+ manual fallback)
3. Moving average overlay on price data
4. Volatility measurement from True Range
5. Classic single-candle patterns


In [ ]:
# ── OHLC Dataset: Fictional Stock "VISU Corp" (60 Trading Days) ───────────
# We simulate 60 business days of price data using a random walk model.
# Mathematical constraints MUST hold:  Low ≤ Open, Close ≤ High  (always)

np.random.seed(2024)

# pd.bdate_range = business-day date range (Mon–Fri, no weekends)
days = pd.bdate_range(start="2024-01-02", periods=60)

# Random walk: each day's return drawn from normal distribution
# loc=0.001 → slight upward drift; scale=0.018 → 1.8% daily volatility
daily_returns = np.random.normal(loc=0.001, scale=0.018, size=60)

# Cumulative product converts daily returns to price levels (starts at $100)
close_prices = 100.0 * np.cumprod(1 + daily_returns)

# Open = previous day's Close (gap-less model)
open_prices = np.roll(close_prices, 1)
open_prices[0] = 100.0    # first day opens at $100

# Intraday range determines wick/body sizes
intraday_range = close_prices * np.random.uniform(0.005, 0.025, 60)
high_prices = (np.maximum(open_prices, close_prices)
               + intraday_range * np.random.uniform(0.4, 1.0, 60))
low_prices  = (np.minimum(open_prices, close_prices)
               - intraday_range * np.random.uniform(0.4, 1.0, 60))

df_ohlc = pd.DataFrame({
    "Open":   open_prices.round(2),
    "High":   high_prices.round(2),
    "Low":    low_prices.round(2),
    "Close":  close_prices.round(2),
    "Volume": np.random.randint(500_000, 3_000_000, 60)
}, index=days)
df_ohlc.index.name = "Date"

# Data quality assertion — OHLC constraints must hold
assert (df_ohlc["High"] >= df_ohlc["Open"]).all(),  "High >= Open violated"
assert (df_ohlc["High"] >= df_ohlc["Close"]).all(), "High >= Close violated"
assert (df_ohlc["Low"]  <= df_ohlc["Open"]).all(),  "Low <= Open violated"
assert (df_ohlc["Low"]  <= df_ohlc["Close"]).all(), "Low <= Close violated"
print("OHLC constraints validated — all good!")

print("\nFirst 10 trading days:")
print(df_ohlc.head(10).to_string())
bullish = (df_ohlc["Close"] > df_ohlc["Open"]).sum()
print(f"\nBullish days: {bullish}/60  ({bullish/60*100:.1f}%)")
print(f"Total return: {(df_ohlc['Close'].iloc[-1]/100-1)*100:.1f}%")


In [ ]:
# ── Candlestick Chart ───────────────────────────────────────────────────────
# mplfinance.plot() handles OHLC rendering professionally.
# The DataFrame MUST:  (a) have columns Open,High,Low,Close (case-sensitive)
#                      (b) have a DatetimeIndex

if MPLFINANCE:
    # ── Style ──────────────────────────────────────────────────────────────────
    style = mpf.make_mpf_style(
        base_mpf_style="charles",       # 'charles' = clean black-and-white look
        rc={"font.size": 10},
        gridcolor="lightgray", gridstyle="--"
    )

    # ── Moving average overlays ────────────────────────────────────────────────
    # mpf.make_addplot() adds any Series as an overlay panel
    ma10 = mpf.make_addplot(df_ohlc["Close"].rolling(10).mean(),
                             color="orange", linewidth=1.8, label="10-Day MA")
    ma20 = mpf.make_addplot(df_ohlc["Close"].rolling(20).mean(),
                             color="dodgerblue", linewidth=1.8,
                             linestyle="--", label="20-Day MA")

    # ── Plot ───────────────────────────────────────────────────────────────────
    # type="candle"  → candlestick rendering (vs "ohlc", "line", "renko")
    # volume=True    → adds a volume histogram panel below the candles
    # returnfig=True → returns (fig, axes) so we can add a legend
    fig, axes = mpf.plot(
        df_ohlc,
        type="candle",
        style=style,
        volume=True,
        addplot=[ma10, ma20],
        figsize=(14, 8),
        title="VISU Corp — 60-Day Candlestick Chart (Jan–Mar 2024)",
        ylabel="Price ($)", ylabel_lower="Volume",
        returnfig=True
    )
    axes[0].legend(["10-Day MA", "20-Day MA"], loc="upper left", fontsize=10)
    plt.tight_layout(); plt.show()

else:
    # ── Manual fallback: draw candles with bar() + plot() ─────────────────────
    print("Drawing manual OHLC chart (install mplfinance for full candlestick support)")
    fig, ax = plt.subplots(figsize=(14, 7))

    for i, (date, row) in enumerate(df_ohlc.iterrows()):
        color       = "green" if row["Close"] >= row["Open"] else "red"
        body_bottom = min(row["Open"], row["Close"])
        body_height = abs(row["Close"] - row["Open"])
        # Body (rectangle)
        ax.bar(i, body_height, bottom=body_bottom, color=color, width=0.7, alpha=0.85)
        # Wick (thin vertical line)
        ax.plot([i, i], [row["Low"], row["High"]], color=color, linewidth=0.9)

    # 10-day MA overlay
    ma_vals = df_ohlc["Close"].rolling(10).mean().values
    ax.plot(range(60), ma_vals, color="orange", linewidth=2, label="10-Day MA")

    tick_pos = range(0, 60, 10)
    ax.set_xticks(list(tick_pos))
    ax.set_xticklabels([df_ohlc.index[i].strftime("%b %d") for i in tick_pos], rotation=30)
    ax.set_title("VISU Corp — 60-Day OHLC Chart (Fallback)", fontsize=15)
    ax.set_xlabel("Trading Day"); ax.set_ylabel("Price ($)")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()

print("\nReading the chart:")
print("  • Green candles → price closed HIGHER than it opened (buyers dominated)")
print("  • Red candles   → price closed LOWER  than it opened (sellers dominated)")
print("  • Long upper wick → buyers pushed price up; sellers pushed back hard")
print("  • Long lower wick → sellers drove price low; buyers stepped in to recover")


In [ ]:
# ── Volatility Analysis from OHLC ──────────────────────────────────────────
# Volatility measures how much prices move in a given period.
# True Range = High - Low  (entire intraday price range, ignoring gaps)
# Rolling volatility = standard deviation of % returns (annualised)

df_ohlc["Body"]      = (df_ohlc["Close"] - df_ohlc["Open"]).abs()
df_ohlc["TrueRange"] = df_ohlc["High"] - df_ohlc["Low"]
df_ohlc["UpperWick"] = df_ohlc["High"] - df_ohlc[["Open","Close"]].max(axis=1)
df_ohlc["LowerWick"] = df_ohlc[["Open","Close"]].min(axis=1) - df_ohlc["Low"]
df_ohlc["Return"]    = df_ohlc["Close"].pct_change()
# Annualised rolling vol: std of returns × sqrt(252 trading days/year)
df_ohlc["Vol_10d"]   = df_ohlc["Return"].rolling(10).std() * np.sqrt(252) * 100

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

# ── Panel 1: True Range ────────────────────────────────────────────────────
axes[0].fill_between(range(60), df_ohlc["TrueRange"],
                     color="steelblue", alpha=0.6, label="True Range (H−L)")
axes[0].plot(range(60), df_ohlc["TrueRange"].rolling(5).mean(),
             color="darkblue", linewidth=2, label="5-Day Avg True Range")
axes[0].set_ylabel("True Range ($)"); axes[0].legend(fontsize=10)
axes[0].set_title("Volatility Analysis — VISU Corp", fontsize=14, pad=10)
axes[0].grid(axis="y", alpha=0.3)

# ── Panel 2: Body vs Wick ─────────────────────────────────────────────────
axes[1].bar(range(60), df_ohlc["Body"],
            color="teal", alpha=0.7, label="Candle Body", width=0.9)
axes[1].bar(range(60), df_ohlc["UpperWick"] + df_ohlc["LowerWick"],
            bottom=df_ohlc["Body"], color="coral", alpha=0.7,
            label="Total Wick", width=0.9)
axes[1].set_ylabel("Price Range ($)"); axes[1].legend(fontsize=10)
axes[1].grid(axis="y", alpha=0.3)

# ── Panel 3: Annualised rolling volatility ────────────────────────────────
axes[2].fill_between(range(60), df_ohlc["Vol_10d"],
                     color="crimson", alpha=0.55, label="10-Day Rolling Vol (ann.%)")
axes[2].axhline(df_ohlc["Vol_10d"].mean(), color="darkred",
                linestyle="--", linewidth=1.5,
                label=f"Mean: {df_ohlc['Vol_10d'].mean():.1f}%")
axes[2].set_ylabel("Volatility (%)"); axes[2].set_xlabel("Trading Day")
axes[2].legend(fontsize=10); axes[2].grid(axis="y", alpha=0.3)

tick_pos = range(0, 60, 10)
axes[2].set_xticks(list(tick_pos))
axes[2].set_xticklabels([df_ohlc.index[i].strftime("%b %d") for i in tick_pos], rotation=30)

plt.tight_layout(); plt.show()

print("Volatility Takeaways:")
print("  • Tall True Range bars → large intraday swings (high volatility)")
print("  • Long wicks > body    → indecision / potential reversal signal")
print("  • Rising 10-day vol    → increasing uncertainty in the market")


In [ ]:
# ── Classic Single-Candle Patterns (Schematic) ────────────────────────────
# These patterns are used by traders to identify potential trend shifts.
# IMPORTANT: Always confirm with multiple sessions and additional context.

fig, axes = plt.subplots(1, 4, figsize=(14, 5))
fig.suptitle("Classic Candlestick Patterns", fontsize=14, y=1.02)

patterns = {
    "Bullish\n(Rising)":    {"o":10.0,"h":12.0,"l":9.5, "c":11.8,"color":"green"},
    "Bearish\n(Falling)":   {"o":11.8,"h":12.0,"l":9.5, "c":10.0,"color":"red"},
    "Doji\n(Indecision)":   {"o":11.0,"h":12.0,"l":9.5, "c":11.05,"color":"gray"},
    "Hammer\n(Reversal?)":  {"o":11.5,"h":11.7,"l":9.0, "c":11.6, "color":"green"},
}

for ax, (name, p) in zip(axes, patterns.items()):
    body_bot = min(p["o"], p["c"])
    body_h   = max(abs(p["c"] - p["o"]), 0.02)   # minimum visible body
    # Body rectangle
    ax.bar(0.5, body_h, bottom=body_bot, color=p["color"],
           width=0.4, edgecolor="black", linewidth=1.2)
    # Lower wick
    ax.plot([0.5, 0.5], [p["l"], body_bot], color="black", linewidth=1.5)
    # Upper wick
    ax.plot([0.5, 0.5], [body_bot + body_h, p["h"]], color="black", linewidth=1.5)
    # Labels
    ax.text(0.5, p["h"]+0.1, f"H:{p['h']}", ha="center", fontsize=8)
    ax.text(0.5, p["l"]-0.18,f"L:{p['l']}", ha="center", fontsize=8)
    ax.text(0.5, max(p["o"],p["c"])+0.05, f"C:{p['c']}", ha="center", fontsize=8, color="navy")
    ax.text(0.5, min(p["o"],p["c"])-0.15, f"O:{p['o']}", ha="center", fontsize=8, color="navy")
    ax.set_title(name, fontsize=12, pad=8)
    ax.set_xlim(0,1); ax.set_ylim(8,13)
    ax.set_xticks([]); ax.yaxis.set_visible(False)
    sns.despine(ax=ax, left=True, bottom=True)

plt.tight_layout(); plt.show()

print("Pattern Guide:")
print("  Bullish : Close > Open — buyers dominated the full session")
print("  Bearish : Close < Open — sellers dominated the full session")
print("  Doji    : Open ≈ Close — no net direction; possible reversal signal")
print("  Hammer  : Long lower wick, small body near top")
print("            Sellers pushed price down; buyers recovered strongly")
print("            Often signals bullish reversal after a downtrend")
print("\n⚠️  Never make trading decisions based on a single candle pattern alone.")


---
# Summary & Key Takeaways

## Time Series Line Plots
- **Date parsing**: `pd.to_datetime()` → `set_index()` → `sort_index()`
- **Axis formatting**: always use `mdates.DateFormatter` — never raw timestamps
- **Rolling averages**: `df[col].rolling(N).mean()` — short window = responsive; long = trend
- **Seasonality**: `axvspan()` for shading; `annotate()` for peak labelling
- **Sort first**: unsorted dates produce meaningless zig-zag lines

## Candlestick Charts
- OHLC encodes **4 prices** per period; body shows direction, wicks show intraday range
- **Green** = Close > Open (bullish); **Red** = Close < Open (bearish)
- **Long wicks** → volatility and indecision; **Doji** = Open ≈ Close
- `mplfinance.plot()` for professional charts; manual `bar()` + `plot()` as fallback
- True Range = H − L; annualised vol = rolling std(returns) × √252

## Decision Guide
| Situation | Recommended Chart |
|-----------|-------------------|
| Monthly/quarterly trend | Line + 12-Period Rolling MA |
| Noisy daily data | Line + Short MA (3–7 periods) |
| Recurring seasonal patterns | Line + `axvspan()` shading |
| Financial OHLC price action | Candlestick (`mplfinance`) |
| Volatility analysis | TrueRange bar chart + rolling vol |

*Next class: Proportions & Part-to-Whole (Pie, Donut, Treemap).*


---
# Guided Lab — Build It Yourself

## Task 1 — Time Series Line Plot

**Scenario:** You analyse monthly website traffic for a news portal over 2 years.

### Steps
1. Use the starter dataset (monthly page-views with seasonal spikes)
2. Plot the raw line with proper date axis formatting
3. Overlay a 3-month and 6-month rolling average
4. Shade months where traffic exceeded **1 million page-views** in gold
5. **Discuss:** Is there a clear trend? What could cause the January spikes?

---

## Task 2 — Candlestick Chart

**Scenario:** You have 30 trading days of OHLC data for "DATAVIZ Inc."

### Steps
1. Use the provided starter OHLC DataFrame
2. Plot with `mplfinance` (or the manual fallback if unavailable)
3. Add a 5-day moving average overlay
4. Find and annotate the day with the largest True Range (highest volatility)
5. **Discuss:** Was the stock trending up or down? Which days showed indecision?


In [ ]:
# ── YOUR CODE: Task 1 — Website Traffic Time Series ───────────────────────

np.random.seed(55)

# Starter dataset — monthly page-views for a news portal
dates_t = pd.date_range("2023-01-01", "2024-12-01", freq="MS")   # 24 months
base_t  = np.linspace(600_000, 950_000, 24)
seasonal_t = np.array([1.25 if m==1 else (1.15 if m==9 else 1.0)
                        for m in dates_t.month])
noise_t    = np.random.normal(0, base_t * 0.06)
pageviews  = (base_t * seasonal_t + noise_t).clip(400_000).round(0)

df_traffic = pd.DataFrame({"PageViews": pageviews}, index=dates_t)
df_traffic.index.name = "Month"
print("Traffic dataset (first 6 rows):")
print(df_traffic.head(6).to_string())

# ── YOUR PLOTTING CODE BELOW ──────────────────────────────────────────────────

# Step 1: Calculate rolling averages
df_traffic["MA_3"] = df_traffic["PageViews"].rolling(window=___).mean()   # fill in
df_traffic["MA_6"] = df_traffic["PageViews"].rolling(window=___).mean()   # fill in

fig, ax = plt.subplots(figsize=(13, 6))

# Step 2: Raw line
ax.plot(df_traffic.index, df_traffic["PageViews"],
        color=___, linewidth=1.5, marker="o", markersize=4,
        label="Monthly PageViews")

# Step 3: Rolling averages
ax.plot(df_traffic.index, df_traffic["MA_3"],
        color="darkorange", linewidth=2, linestyle="--", label="3-Month MA")
ax.plot(df_traffic.index, df_traffic["MA_6"],
        color="crimson", linewidth=2.5, label="6-Month MA")

# Step 4: Shade months exceeding 1,000,000 page-views
# HINT: loop over the index and use ax.axvspan() for months above threshold
threshold = 1_000_000
for date, row in df_traffic.iterrows():
    if row["PageViews"] > threshold:
        ax.axvspan(date - pd.Timedelta(days=15),
                   date + pd.Timedelta(days=15),
                   alpha=___, color="gold")   # try 0.3

# Step 5: Axis formatting
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=___))   # every N months
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=30, ha="right")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e6:.2f}M"))

ax.set_title("___ YOUR TITLE HERE ___", fontsize=15)
ax.set_xlabel("Month"); ax.set_ylabel("Page Views")
ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── YOUR CODE: Task 2 — DATAVIZ Inc. Candlestick Chart ────────────────────

np.random.seed(88)

# Starter OHLC dataset — 30 trading days
days_dv  = pd.bdate_range("2024-03-01", periods=30)
ret_dv   = np.random.normal(0.002, 0.016, 30)
close_dv = 50.0 * np.cumprod(1 + ret_dv)
open_dv  = np.roll(close_dv, 1); open_dv[0] = 50.0
rng_dv   = close_dv * np.random.uniform(0.006, 0.022, 30)
high_dv  = np.maximum(open_dv, close_dv) + rng_dv * np.random.uniform(0.3, 1.0, 30)
low_dv   = np.minimum(open_dv, close_dv) - rng_dv * np.random.uniform(0.3, 1.0, 30)

df_dv = pd.DataFrame({
    "Open":  open_dv.round(2),  "High": high_dv.round(2),
    "Low":   low_dv.round(2),   "Close": close_dv.round(2),
    "Volume": np.random.randint(200_000, 1_500_000, 30)
}, index=days_dv)
df_dv.index.name = "Date"

df_dv["TrueRange_dv"] = df_dv["High"] - df_dv["Low"]
peak_date = df_dv["TrueRange_dv"].idxmax()
print(f"Highest volatility: {peak_date.date()}  TrueRange=${df_dv.loc[peak_date,'TrueRange_dv']:.2f}")

# ── YOUR PLOTTING CODE BELOW ──────────────────────────────────────────────────

if MPLFINANCE:
    ma5_overlay = mpf.make_addplot(
        df_dv["Close"].rolling(___).mean(),   # fill window
        color="orange", linewidth=2, label="5-Day MA"
    )
    mpf.plot(
        df_dv[["Open","High","Low","Close","Volume"]],
        type=___,       # "candle" or "ohlc"
        volume=True,
        addplot=[ma5_overlay],
        figsize=(13, 7),
        title="DATAVIZ Inc. — 30-Day Candlestick Chart",
        style=___       # try "yahoo", "charles", or "mike"
    )
else:
    fig, ax = plt.subplots(figsize=(13, 6))
    for i, (date, row) in enumerate(df_dv.iterrows()):
        c = "green" if row["Close"] >= row["Open"] else "red"
        ax.bar(i, abs(row["Close"]-row["Open"]),
               bottom=min(row["Open"],row["Close"]), color=c, width=0.7, alpha=0.85)
        ax.plot([i,i],[row["Low"],row["High"]], color=c, linewidth=0.9)
    # 5-day MA
    ma5v = df_dv["Close"].rolling(5).mean().values
    ax.plot(range(30), ma5v, color="orange", linewidth=2, label="5-Day MA")
    # Annotate peak volatility day
    pk = df_dv.index.get_loc(peak_date)
    ax.annotate("Peak Vol!", xy=(pk, df_dv.loc[peak_date,"High"]),
                xytext=(pk+1, df_dv.loc[peak_date,"High"]+0.5),
                arrowprops=dict(arrowstyle="->",color="purple"),
                color="purple", fontsize=9)
    ax.set_xticks(range(0,30,5))
    ax.set_xticklabels([df_dv.index[i].strftime("%b %d") for i in range(0,30,5)], rotation=30)
    ax.set_title("DATAVIZ Inc. — 30-Day OHLC (Fallback)", fontsize=14)
    ax.set_xlabel("Trading Day"); ax.set_ylabel("Price ($)")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()


---
# Challenge

**Scenario:** Compare two competing company stocks — ALPHA Corp and BETA Corp — over 6 months.

## Your Tasks
1. Generate OHLC data for both companies using `np.random.normal()`:
   - ALPHA: starts at $80, daily vol 1.6% (moderate)
   - BETA: starts at $120, daily vol 2.5% (high)
2. **Dual close-price chart**: plot both Close price lines on the same axes with clear legend
3. **ALPHA candlestick**: plot 30 days of ALPHA's candles with a 20-day MA overlay
4. **Volatility summary**: print a table comparing average TrueRange per week for both stocks

*Bonus:* Plot the price ratio ALPHA/BETA as a third panel.
When the ratio rises, ALPHA outperforms BETA. What do you observe?


In [ ]:
# ── Challenge Starter ──────────────────────────────────────────────────────────

def make_ohlc(seed, n_days, days_idx, start_price, drift, vol):
    """Generate a synthetic OHLC DataFrame."""
    np.random.seed(seed)
    rets   = np.random.normal(drift, vol, n_days)
    closes = start_price * np.cumprod(1 + rets)
    opens  = np.roll(closes, 1); opens[0] = start_price
    rng    = closes * np.random.uniform(0.005, 0.03, n_days)
    highs  = np.maximum(opens, closes) + rng * np.random.uniform(0.3, 1.0, n_days)
    lows   = np.minimum(opens, closes) - rng * np.random.uniform(0.3, 1.0, n_days)
    return pd.DataFrame({
        "Open": opens.round(2), "High": highs.round(2),
        "Low":  lows.round(2),  "Close": closes.round(2),
        "Volume": np.random.randint(300_000, 2_000_000, n_days)
    }, index=days_idx)

trading_days = pd.bdate_range("2024-01-02", "2024-06-28")
n = len(trading_days)

df_alpha = make_ohlc(42, n, trading_days, 80,  0.001,  0.016)
df_beta  = make_ohlc(77, n, trading_days, 120, 0.0005, 0.025)
df_alpha.index.name = df_beta.index.name = "Date"

print("ALPHA Close summary:"); print(df_alpha["Close"].describe().round(2))
print("\nBETA Close summary:");  print(df_beta["Close"].describe().round(2))

# ── WRITE YOUR ANALYSIS CODE BELOW ───────────────────────────────────────────
# 1. Dual close-price line chart (both on same axes)
# 2. ALPHA candlestick with 20-day MA
# 3. Volatility comparison table (TrueRange by week)
# Bonus: ALPHA/BETA ratio panel


---
# Homework — Temperature Trend Analysis

## Dataset to Create
Build a synthetic **daily temperature dataset** for a city over **2 full years (2023–2024)**.

| Variable | Type | Description |
|----------|------|-------------|
| `Temp_C` | float | Daily temperature — realistic seasonal cosine curve + noise |
| `Temp_F` | float | Fahrenheit: `Temp_C × 9/5 + 32` |
| `Rain_mm`| float | Daily rainfall — higher in spring and autumn |

## Required Deliverables
1. **Time Series Plot** (°C) with:
   - Raw daily line (thin, semi-transparent)
   - 30-day rolling average (bold — the seasonal trend)
   - Winter (Dec, Jan, Feb) shaded **light blue**
   - Summer (Jun, Jul, Aug) shaded **light orange**
   - Date axis every 2 months
2. **Dual-panel figure** (`sharex=True`):
   - Top: temperature line
   - Bottom: rainfall as a `bar()` chart
3. **Markdown cell** answering:
   - Approximate seasonal amplitude (max summer avg − min winter avg)?
   - Is there a year-on-year warming trend? How can you tell?
   - Which month typically has the highest rainfall?

### Bonus
- 7-day rolling average on the rainfall panel
- Mark days where `Temp_C > 35` with a red `axvline()`

**Submit:** completed `.ipynb` with code + markdown explanations. 🌡️


In [ ]:
# ── Homework Starter Code ──────────────────────────────────────────────────
np.random.seed(2025)

dates_hw = pd.date_range("2023-01-01", "2024-12-31", freq="D")
n_hw     = len(dates_hw)

# Seasonal temperature using cosine wave
# Northern hemisphere: coldest ~Jan (day ~15), warmest ~Jul (day ~196)
doy = np.array([d.timetuple().tm_yday for d in dates_hw])
seasonal_temp = 15 + 13 * np.cos(2 * np.pi * (doy - 200) / 365)
# 15°C annual mean; ±13°C amplitude; phase shift to peak in July

daily_var     = np.random.normal(0, 3.5, n_hw)
warming       = np.linspace(0, 0.8, n_hw)   # +0.8°C warming over 2 yrs
temp_c        = seasonal_temp + daily_var + warming

# Rainfall: higher in spring (Mar-May) and autumn (Sep-Nov)
rain_scale = np.array([
    8 if m in (3,4,5,9,10,11) else (2 if m in (6,7,8) else 5)
    for m in dates_hw.month
])
rain_mm = np.random.exponential(scale=rain_scale)

df_hw = pd.DataFrame({
    "Temp_C":  temp_c.round(1),
    "Temp_F":  (temp_c * 9/5 + 32).round(1),
    "Rain_mm": rain_mm.round(1)
}, index=dates_hw)
df_hw.index.name = "Date"

print("Homework dataset preview:")
print(df_hw.head(8).to_string())
print(f"\nShape: {df_hw.shape}")
print(df_hw.describe().round(2))

# ── YOUR PLOTTING CODE BELOW ─────────────────────────────────────────────────
# Create a 2-panel figure with sharex=True
fig, (ax_t, ax_r) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                   gridspec_kw={"height_ratios": [2, 1]})

# Panel 1: Temperature
ax_t.plot(df_hw.index, df_hw["Temp_C"],
          color="steelblue", linewidth=0.7, alpha=0.5, label="Daily Temp")
roll30 = df_hw["Temp_C"].rolling(___).mean()   # fill: 30
ax_t.plot(df_hw.index, roll30, color="darkred", linewidth=2.2, label="30-Day Avg")

# Shade winters and summers here using ax_t.axvspan()
# ...

ax_t.set_ylabel("Temperature (°C)")
ax_t.set_title("___ YOUR TITLE HERE ___", fontsize=15)
ax_t.legend(); ax_t.grid(axis="y", alpha=0.3)

# Panel 2: Rainfall
ax_r.bar(df_hw.index, df_hw["Rain_mm"], color="cornflowerblue", alpha=0.6, width=1)
ax_r.plot(df_hw.index, df_hw["Rain_mm"].rolling(7).mean(),
          color="navy", linewidth=1.5, label="7-Day Avg")
ax_r.set_ylabel("Rainfall (mm)"); ax_r.legend(); ax_r.grid(axis="y", alpha=0.3)

ax_r.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax_r.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()


---
1. **Time Series:** What does a 12-month rolling average represent, and why prefer it over a 3-month window for detecting long-term trends?
2. **Candlestick:** A candle: Open=$50, Close=$49, High=$54, Low=$47. Bullish or bearish? Which feature shows most volatility?
3. **Reflection:** Name one real-world application of candlestick-style OHLC charts *outside* of finance.


In [ ]:


answer_q1 = """
YOUR ANSWER: A 12-month rolling average...
"""

answer_q2 = """
YOUR ANSWER: The candle is... because...
The most volatile feature is...
"""

answer_q3 = """
YOUR ANSWER: OHLC outside finance could show...
"""

print("=" * 65)
print("EXIT TICKET — Time Series & Candlestick Class")
print("=" * 65)
print(f"Q1:\n{answer_q1.strip()}")
print("-" * 65)
print(f"Q2:\n{answer_q2.strip()}")
print("-" * 65)
print(f"Q3:\n{answer_q3.strip()}")
print("=" * 65)
print("\nExcellent work today! 📈")
